In [1]:
from cgra import *
from kernels import *
import random

In [2]:
kernel_name = "benchmarks/rv-summit/mmul_ws"
version = ""

In [3]:
# Global variables
CGRA_N_ROWS = 4
CGRA_N_COLS = 4
# Adress
first_addr = 20000
BLOCK_SIZE = 3

In [4]:
def printAsMatrix(array, rows, cols):
    for i in range(rows):
        print(array[i * cols:(i + 1) * cols])

In [5]:
def runKernel(load_addrs, store_addrs=[], max_it=1000):
    # Run kernel
    run(kernel_name, pr=["ROUT", "INST"], load_addrs=load_addrs, store_addrs=store_addrs, version=version, limit=max_it)

In [6]:
def getResult(first_addr_C, end_addr_C, rowsA, colsB):
    result = [0 for _ in range(rowsA*colsB)]
    with open( kernel_name + "/memory_out"+version+".csv", 'r') as f:
        csv_reader = csv.reader(f, delimiter=',')
        for row in csv_reader:
            try:
                if(int(row[0]) >= first_addr_C) and (int(row[0]) < end_addr_C):
                    result[int((int(row[0]) - first_addr_C)/4)] = int(row[1])
            except ValueError:
                print("Error: Values in memory_out CSV file are not integers.")
    return result

In [7]:
def getOneElement(addr):
    return (getResult(first_addr_C=addr, end_addr_C=addr + 4, rowsA=1, colsB=1))[0]

In [8]:
def mmul_cpu(A_data, B_data, rowsA, colsA, colsB):
    expected_res = [0 for _ in range(rowsA*colsB)]
    for rA in range(rowsA):
        for cB in range(colsB):
            sum = 0
            for cA in range(colsA):
                sum += A_data[rA*colsA + cA] * B_data[cA*colsB + cB]
            expected_res[rA*colsB + cB] = sum 
    return expected_res

In [9]:
# Test dimensions (4x3x12)
rowsA = 8
colsA = 3
colsB = 12

A_data = [random.randint(-10, 10) for _ in range(rowsA * colsA)]
B_data = [random.randint(-10, 10) for _ in range(colsA * colsB)]
C_data = [0 for _ in range(rowsA * colsB)]

A_data_cpy = A_data.copy()
B_data_cpy = B_data.copy()


In [ ]:
# Tiling needed

# Recorrer B por bloques de 3 filas (nRegistros usables en RCs) y 12 columnas (nRCs computo)
for rB in range(0, colsA, 3):

    for cB in range(0, colsB, 12):
        print(f"Processing block: (rB, cB) = ({rB}, {cB})")
        # Config CGRA for this tile

        # first_addr_A      first_addr_A + 4    first_addr_A + 8    first_addr_A + 12
        # B[rB][cB]         B[rB][cB+1]         B[rB][cB+2]         B[rB][cB+3]
        # B[rB][cB+4]       B[rB][cB+5]         B[rB][cB+6]         B[rB][cB+7]
        # B[rB][cB+8]       B[rB][cB+9]         B[rB][cB+10]        B[rB][cB+11]
        # -----------------------------------------------------------------------
        # 4*4*colsA         4*4*colsA           4*4*colsA           4*4*colsA
        # B[rB+1][cB]       B[rB+1][cB+1]       B[rB+1][cB+2]       B[rB+1][cB+3]
        # B[rB+1][cB+4]     B[rB+1][cB+5]       B[rB+1][cB+6]       B[rB+1][cB+7]
        # B[rB+1][cB+8]     B[rB+1][cB+9]       B[rB+1][cB+10]      B[rB+1][cB+11]
        # ------------------------------------------------------------------------
        # ----------------  nItLoop             -----------------   -----------------
        # B[rB+2][cB]       B[rB+2][cB+1]       B[rB+2][cB+2]       B[rB+2][cB+3]
        # B[rB+2][cB+4]     B[rB+2][cB+5]       B[rB+2][cB+6]       B[rB+2][cB+7]
        # B[rB+2][cB+8]     B[rB+2][cB+9]       B[rB+2][cB+10]      B[rB+2][cB+11]

        nItLoop = rowsA // 4
        first_addr_A = first_addr
        
        config_vals_col0 = [ first_addr_A,
                             B_data[rB*colsB + cB],
                             B_data[rB*colsB + cB + 4],
                             B_data[rB*colsB + cB + 8],
                             4*colsA,
                             B_data[(rB+1)*colsB + cB],
                             B_data[(rB+1)*colsB + cB + 4],
                             B_data[(rB+1)*colsB + cB + 8],
                             B_data[(rB+2)*colsB + cB],
                             B_data[(rB+2)*colsB + cB + 4],
                             B_data[(rB+2)*colsB + cB + 8]
        ]
        print("Config col 0:")
        print(config_vals_col0)
        config_vals_col1 = [ first_addr_A + 4*colsA,
                             B_data[rB*colsB + cB + 1],
                             B_data[rB*colsB + cB + 5],
                             B_data[rB*colsB + cB + 9],
                             4*colsA,
                             B_data[(rB+1)*colsB + cB + 1],
                             B_data[(rB+1)*colsB + cB + 5],
                             B_data[(rB+1)*colsB + cB + 9],
                             nItLoop,
                             B_data[(rB+2)*colsB + cB + 1],
                             B_data[(rB+2)*colsB + cB + 5],
                             B_data[(rB+2)*colsB + cB + 9]
        ]
        print("Config col 1:")
        print(config_vals_col1)
        config_vals_col2 = [ first_addr_A + 2*4*colsA,
                             B_data[rB*colsB + cB + 2],
                             B_data[rB*colsB + cB + 6],
                             B_data[rB*colsB + cB + 10],
                             4*colsA,
                             B_data[(rB+1)*colsB + cB + 2],
                             B_data[(rB+1)*colsB + cB + 6],
                             B_data[(rB+1)*colsB + cB + 10],
                             B_data[(rB+2)*colsB + cB + 2],
                             B_data[(rB+2)*colsB + cB + 6],
                             B_data[(rB+2)*colsB + cB + 10]
        ]
        print("Config col 2:")
        print(config_vals_col2)
        config_vals_col3 = [ first_addr_A + 3*4*colsA,
                            B_data[rB*colsB + cB + 3],
                            B_data[rB*colsB + cB + 7],
                            B_data[rB*colsB + cB + 11],
                            4*colsA,
                            B_data[(rB+1)*colsB + cB + 3],
                            B_data[(rB+1)*colsB + cB + 7],
                            B_data[(rB+1)*colsB + cB + 11],
                            B_data[(rB+2)*colsB + cB + 3],
                            B_data[(rB+2)*colsB + cB + 7],
                            B_data[(rB+2)*colsB + cB + 11]
        ]
        print("Config col 3:")
        print(config_vals_col3)

        addr = 0
        for cfg in [config_vals_col0, config_vals_col1,
                    config_vals_col2, config_vals_col3]:
            kernel_add_memory_region(kernel_name, addr, cfg, version=version)
            addr += len(cfg) * 4

        kernel_add_memory_region(kernel_name, first_addr_A, A_data, version=version)
        kernel_add_memory_region(kernel_name, first_addr_A + rowsA * colsA * 4, B_data, version=version)

        print("Memory written")
        load_addrs = [
            0,
            len(config_vals_col0) * 4,
            (len(config_vals_col0) + len(config_vals_col1)) * 4,
            (len(config_vals_col0) + len(config_vals_col1) + len(config_vals_col2)) * 4
        ]
        
        store_addrs = [ 30000, 40000, 50000, 60000]
        print("Run kernel")
        # Run kernel
        runKernel(load_addrs, store_addrs=store_addrs, max_it=20000)

        print("Get results")

        # Get results and store them into C
        # Col 0: c0, c4, c8
        # r0, r1, r2, r3
        # r4, r5, r6, r7, ....
        rAux = 0
        cAux = cB
        addrCol0 = 30000
        for i in range(nItLoop):
            for j in range(4):
                C_data[rAux*colsB + cAux] += getOneElement(addrCol0)
                addrCol0 += 4
                C_data[rAux*colsB + cAux + 4] += getOneElement(addrCol0)
                addrCol0 += 4
                C_data[rAux*colsB + cAux + 8] += getOneElement(addrCol0)
                addrCol0 += 4
                rAux += 1

        rAux = 0
        rMod = 1
        addrCol1 = 40000
        for i in range(nItLoop):
            for j in range(4):
                C_data[(rAux+ rMod)*colsB + cAux + 1] += getOneElement(addrCol1)
                addrCol1 += 4
                C_data[(rAux+ rMod)*colsB + cAux + 5] += getOneElement(addrCol1)
                addrCol1 += 4
                C_data[(rAux+ rMod)*colsB + cAux + 9] += getOneElement(addrCol1)
                addrCol1 += 4
                rMod += 1
                rMod = rMod % 4
            rAux += 4 

        rAux = 0
        rMod = 2
        addrCol2 = 50000
        for i in range(nItLoop):
            for j in range(4):
                C_data[(rAux+ rMod)*colsB + cAux + 2] += getOneElement(addrCol2)
                addrCol2 += 4
                C_data[(rAux+ rMod)*colsB + cAux + 6] += getOneElement(addrCol2)
                addrCol2 += 4
                C_data[(rAux+ rMod)*colsB + cAux + 10] += getOneElement(addrCol2)
                addrCol2 += 4
                rMod += 1
                rMod = rMod % 4
            rAux += 4 

        rAux = 0
        rMod = 3
        addrCol3 = 60000
        for i in range(nItLoop):
            for j in range(4):
                C_data[(rAux+ rMod)*colsB + cAux + 3] += getOneElement(addrCol3)
                addrCol3 += 4
                C_data[(rAux + rMod)*colsB + cAux + 7] += getOneElement(addrCol3)
                addrCol3 += 4
                C_data[(rAux+ rMod)*colsB + cAux + 11] += getOneElement(addrCol3)
                addrCol3 += 4
                rMod += 1
                rMod = rMod % 4
            rAux += 4 

        print("C:")
        printAsMatrix(C_data, rowsA, colsB)

Processing block: (rB, cB) = (0, 0)
Config col 0:
[20000, -1, 7, -9, 12, -3, 2, 7, 10, -8, 5]
Config col 1:
[20012, -6, -9, 0, 12, 4, 3, -10, 2, 4, -6, -8]
Config col 2:
[20024, -3, 0, 2, 12, -5, 0, -2, 6, -1, 10]
Config col 3:
[20036, -8, -3, -6, 12, -9, -5, 2, 4, 0, -9]
Memory written
Run kernel
Instr =  0 ( 0 )
[20000, 20012, 20024, 20036]    [LWD R0  4, LWD R0  4, LWD R0  4, LWD R0  4]    
[  -1,   -6,   -3,   -8]    [LWD R0  4, LWD R0  4, LWD R0  4, LWD R0  4]    
[   7,   -9,    0,   -3]    [LWD R0  4, LWD R0  4, LWD R0  4, LWD R0  4]    
[  -9,    0,    2,   -6]    [LWD R0  4, LWD R0  4, LWD R0  4, LWD R0  4]    
Aprox cycles this pc: 17
-------
Instr =  1 ( 1 )
[  12,   12,   12,   12]    [LWD R1  4, LWD R1  4, LWD R1  4, LWD R1  4]    
[  -3,    4,   -5,   -9]    [LWD R1  4, LWD R1  4, LWD R1  4, LWD R1  4]    
[   2,    3,    0,   -5]    [LWD R1  4, LWD R1  4, LWD R1  4, LWD R1  4]    
[   7,  -10,   -2,    2]    [LWD R1  4, LWD R1  4, LWD R1  4, LWD R1  4]    
Aprox cycles t

In [11]:
# Get cpu output
expected_res = mmul_cpu(A_data, B_data, rowsA, colsA, colsB)

# Check result correctness
errors = 0
for i in range(len(expected_res)):
    if expected_res[i] != C_data[i]:
        errors += 1
if errors > 0:
    print("Err: " + str(errors) + " out of " + str(rowsA*colsB))
    print("CGRA: ")
    printAsMatrix(C_data, rowsA, colsB)
    print("Expected: ")
    printAsMatrix(expected_res, rowsA, colsB)
    print("A: ")
    printAsMatrix(A_data, rowsA, colsA)
    print("B: ")
    printAsMatrix(B_data, colsA, colsB)
else:
    print("OK")



Err: 47 out of 96
CGRA: 
[13, 44, 19, 43, -48, 51, -1, 13, 66, -18, -4, 29]
[-61, -10, -63, -94, 65, 15, 4, -39, -5, -28, -46, 30]
[-87, -28, -33, 19, 46, 87, 10, 27, -35, 110, -102, 108]
[-44, -52, -52, -94, 84, -48, 3, -34, -73, 4, -18, -17]
[-61, -10, -63, -94, 65, 15, 4, -39, -5, -28, -46, 30]
[-87, -28, -33, 19, 46, 87, 10, 27, -35, 110, -102, 108]
[-44, -52, -52, -94, 84, -48, 3, -34, -73, 4, -18, -17]
[30, 42, -18, -81, -9, -36, -6, -54, 66, -138, 48, -54]
Expected: 
[13, 44, 19, 43, -48, 51, -1, 13, 66, -18, -4, 29]
[-61, -10, -63, -94, 65, 15, 4, -39, -5, -28, -46, 30]
[-87, -28, -33, 19, 46, 87, 10, 27, -35, 110, -102, 108]
[-44, -52, -52, -94, 84, -48, 3, -34, -73, 4, -18, -17]
[30, 42, -18, -81, -9, -36, -6, -54, 66, -138, 48, -54]
[-17, 2, 37, 124, -39, 87, 5, 67, 0, 120, -52, 83]
[46, 48, -14, -90, -16, -54, -8, -62, 74, -164, 68, -76]
[88, 100, 48, 34, -110, 12, -10, -12, 146, -140, 76, -42]
A: 
[-6, 1, 1]
[3, 6, -4]
[-4, -3, -10]
[8, 2, -3]
[3, 9, 6]
[-9, -8, -5]
[4, 10